In [44]:
"""
Configuration constants for the content management system.
DO NOT MODIFY THIS FILE.
"""

import os

# API Configuration for OpenAI
API_CONFIG = {
    "api_key": os.environ.get("OPENAI_API_KEY", ""),
    "base_url": os.environ.get("OPENAI_API_BASE", ""),
    "model": "gpt-5-mini"
}

# Model pricing per 1M tokens (input and output rates differ)
MODEL_PRICING = {
    "gpt-5-mini": {
        "input": 0.80,    # $0.80 per 1M input tokens
        "output": 1.20,   # $1.20 per 1M output tokens
        "max_tokens": 128000,
        "complexity_range": (1, 5)  # Can handle tasks with complexity 1-5
    },
    "gpt-5": {
        "input": 25.0,    # $25 per 1M input tokens
        "output": 50.0,   # $50 per 1M output tokens
        "max_tokens": 128000,
        "complexity_range": (1, 10)  # Can handle any task complexity
    },
    "claude-sonnet-4": {
        "input": 3.0,     # $3 per 1M input tokens
        "output": 15.0,   # $15 per 1M output tokens
        "max_tokens": 200000,
        "complexity_range": (1, 8)  # Can handle tasks with complexity 1-8
    }
}

# Tokenizer encodings for different model families
MODEL_ENCODINGS = {
    "gpt-5-mini": "o200k_base",
    "gpt-5": "o200k_base",
    "claude-sonnet-4": "cl100k_base"  # Approximate using this encoding
}

# Content type specific settings
CHUNK_OVERLAP_PERCENTAGE = 0.15  # 15% overlap between chunks

# Section headers for different content types
RESEARCH_PAPER_SECTIONS = [
    "Abstract",
    "Introduction",
    "Related Work",
    "Method",
    "Methods",
    "Methodology",
    "Results",
    "Discussion",
    "Conclusion",
    "References"
]

In [ ]:
"""
Token Optimizer Module

This module handles token counting, cost estimation, and dynamic model selection
to optimize AI API usage costs while meeting task requirements.

COMPLETE THE TODO SECTIONS IN THIS FILE.
"""

import tiktoken
from typing import Dict
from config import MODEL_PRICING, MODEL_ENCODINGS


class TokenOptimizer:
    """
    Optimizes token usage and costs for AI API calls.

    This class provides utilities for:
    - Counting tokens accurately for different models
    - Estimating costs before making API calls
    - Selecting the optimal model based on task complexity and budget
    """

    def __init__(self):
        """Initialize the TokenOptimizer with model pricing configuration."""
        self.model_pricing = MODEL_PRICING
        self.model_encodings = MODEL_ENCODINGS

    def count_tokens(self, text: str, model: str) -> int:
        """
        Count the number of tokens in text using the correct tokenizer for the model.

        This is critical for accurate cost estimation. Different model families use
        different tokenizers, so the same text will have different token counts.

        Args:
            text: The text to tokenize
            model: The model name (e.g., "gpt-5-mini", "gpt-5", "claude-sonnet-4")

        Returns:
            Number of tokens in the text

        Raises:
            ValueError: If model is not recognized

        Hints:
        1. Get the correct encoding for the model from self.model_encodings
        2. If model not in encodings, raise ValueError
        3. Use tiktoken.get_encoding() to get the tokenizer
        4. Encode the text and return the length
        5. Handle empty strings (return 0)

        Example:
            optimizer = TokenOptimizer()
            count = optimizer.count_tokens("Hello world", "gpt-5-mini")
            # Returns: 2
        """
        encoding = self.model_encodings.get(model)

        if not encoding:
            raise ValueError(f"Unknown model: {model}")

        tokenizer = tiktoken.get_encoding(encoding)
        tokens = tokenizer.encode(text)
        return len(tokens) if tokens else 0

    def estimate_cost(
        self,
        text: str,
        model: str,
        estimated_response_tokens: int
    ) -> float:
        """
        Estimate the total cost of an API call including input and output tokens.

        Different models charge different rates for input vs output tokens.
        For example, GPT-5 charges $25/1M input tokens but $50/1M output tokens.

        Args:
            text: The prompt text (input)
            model: The model name
            estimated_response_tokens: Expected number of tokens in the response

        Returns:
            Estimated cost in dollars (e.g., 0.0023 for $0.0023)

        Raises:
            ValueError: If model is not recognized

        Hints:
        1. Count the input tokens using self.count_tokens()
        2. Get pricing from self.model_pricing (check if model exists)
        3. Calculate input cost: (input_tokens / 1_000_000) * input_price
        4. Calculate output cost: (estimated_response_tokens / 1_000_000) * output_price
        5. Return total cost (input + output)
        6. Prices in config are per 1M tokens

        Example:
            optimizer = TokenOptimizer()
            cost = optimizer.estimate_cost("Long prompt", "gpt-5-mini", 500)
            # Returns: ~0.0012 (varies based on actual token count)
        """
        tokens = self.count_tokens(text, model)
        pricing = self.model_pricing.get(model)

        if not pricing:
            raise ValueError(f"Unknown model: {model}")

        input_cost = (tokens / 1_000_000) * pricing["input"]
        output_cost = (estimated_response_tokens / 1_000_000) * pricing["output"]

        return input_cost + output_cost

    def select_optimal_model(
        self,
        task_complexity: int,
        max_cost_per_call: float
    ) -> str:
        """
        Select the best model that meets task complexity needs within budget.

        Task complexity ranges from 1-10:
        - 1-3: Simple tasks (classification, extraction)
        - 4-6: Medium tasks (summarization, basic analysis)
        - 7-10: Complex tasks (reasoning, multi-step analysis)

        Strategy:
        - Find all models that can handle the complexity
        - Among those, prefer cheaper models
        - Ensure the model's typical cost is under budget

        Args:
            task_complexity: Task complexity score (1-10)
            max_cost_per_call: Maximum acceptable cost per API call

        Returns:
            Model name that best fits requirements

        Raises:
            ValueError: If no model can meet both requirements

        Hints:
        1. Iterate through models in self.model_pricing
        2. Check if complexity is in model's complexity_range
        3. Estimate a typical cost (assume 1000 input + 500 output tokens)
        4. Keep models that are under max_cost_per_call
        5. Among valid models, return the cheapest (lowest input cost)
        6. If no model works, raise ValueError with clear message
        7. Prefer models in order: gpt-5-mini, claude-sonnet-4, gpt-5

        Example:
            optimizer = TokenOptimizer()
            model = optimizer.select_optimal_model(complexity=3, max_cost=0.01)
            # Returns: "gpt-5-mini" (cheap and sufficient for complexity 3)

            model = optimizer.select_optimal_model(complexity=9, max_cost=0.50)
            # Returns: "gpt-5" (only model that can handle complexity 9)
        """
        valid_models = []
        for model, pricing in self.model_pricing.items():
            if pricing["complexity_range"][0] <= task_complexity <= pricing["complexity_range"][1]:
                typical_cost = self.estimate_cost(" ".join(["Long prompt"] * 1000), model, 500)
                if typical_cost <= max_cost_per_call:
                    valid_models.append((model, typical_cost))

        if not valid_models:
            raise ValueError("No suitable model found.")

        valid_models.sort(key=lambda x: x[1])  # Sort by estimated cost

        return valid_models[0][0]

In [91]:
"""
Content Chunker Module

This module implements intelligent document chunking strategies that respect
content structure and maintain semantic coherence across chunk boundaries.
"""

from typing import List
# from config import CHUNK_OVERLAP_PERCENTAGE, RESEARCH_PAPER_SECTIONS
CHUNK_OVERLAP_PERCENTAGE = 0.15
RESEARCH_PAPER_SECTIONS = ["Introduction", "Methods", "Results", "Discussion"]


class ContentChunker:
    """
    Intelligently chunks large documents based on content type and structure.

    Different content types require different chunking strategies:
    - Research papers: Split by sections
    - Conversations: Maintain speaker exchanges
    - Code: Keep functions and classes intact
    - Markdown: Split at heading boundaries
    """

    def __init__(self):
        """Initialize the ContentChunker with configuration."""
        self.overlap_percentage = CHUNK_OVERLAP_PERCENTAGE
        self.research_sections = RESEARCH_PAPER_SECTIONS

    def calculate_overlap_size(self, max_tokens: int) -> int:
        """
        Calculate appropriate overlap size for maintaining context between chunks.
        Standard practice is 10-20% overlap, we use 15%.
        """
        return int(round(max_tokens * self.overlap_percentage))

    def chunk_by_content_type(
        self, content: str, content_type: str, max_tokens: int
    ) -> List[str]:
        """
        Chunk content based on its type, respecting natural boundaries.
        """
        if not content or not content.strip():
            return []

        overlap_size = self.calculate_overlap_size(max_tokens)

        if content_type == "research_paper":
            return self._chunk_research_paper(content, max_tokens, overlap_size)
        elif content_type == "conversation":
            return self._chunk_conversation(content, max_tokens, overlap_size)
        elif content_type == "code":
            return self._chunk_code(content, max_tokens, overlap_size)
        elif content_type == "markdown":
            return self._chunk_markdown(content, max_tokens, overlap_size)
        else:
            raise ValueError(f"Unknown content type: {content_type}")

    def _chunk_research_paper(
        self, content: str, max_tokens: int, overlap_size: int
    ) -> List[str]:
        """
        Chunk research paper by section headers safely fallback to double newlines.
        """
        max_chars = max_tokens * 4

        lines = content.split("\n")
        sections = []
        current_section_lines = []

        for line in lines:
            if line.strip() in self.research_sections:
                if current_section_lines:
                    sections.append("\n".join(current_section_lines))
                current_section_lines = [line]
            else:
                current_section_lines.append(line)

        if current_section_lines:
            sections.append("\n".join(current_section_lines))

        raw_chunks = []
        current_chunk = ""

        for section_text in sections:
            if (
                len(current_chunk) + (1 if current_chunk else 0) + len(section_text)
                <= max_chars
            ):
                current_chunk += ("\n" if current_chunk else "") + section_text
            else:
                if current_chunk:
                    raw_chunks.append(current_chunk.strip())
                    current_chunk = ""

                if len(section_text) > max_chars:
                    paragraphs = section_text.split("\n\n")
                    for paragraph in paragraphs:
                        if (
                            len(current_chunk)
                            + (2 if current_chunk else 0)
                            + len(paragraph)
                            <= max_chars
                        ):
                            current_chunk += (
                                "\n\n" if current_chunk else ""
                            ) + paragraph
                        else:
                            if current_chunk:
                                raw_chunks.append(current_chunk.strip())
                            current_chunk = paragraph
                else:
                    current_chunk = section_text

        if current_chunk:
            raw_chunks.append(current_chunk.strip())

        final_chunks = []
        for i, chunk in enumerate(raw_chunks):
            if i == 0:
                final_chunks.append(chunk)
            else:
                overlap_text = raw_chunks[i - 1][-overlap_size:]
                final_chunks.append(overlap_text + chunk)

        return final_chunks

    def _chunk_conversation(
        self, content: str, max_tokens: int, overlap_size: int
    ) -> List[str]:
        """
        Chunk conversation maintaining semantic speaker exchanges cleanly.
        """
        max_chars = max_tokens * 4

        exchanges = content.split("\n\n")
        raw_chunks = []
        current_chunk = ""

        for exchange in exchanges:
            if (
                len(current_chunk) + (2 if current_chunk else 0) + len(exchange)
                <= max_chars
            ):
                current_chunk += ("\n\n" if current_chunk else "") + exchange
            else:
                if current_chunk:
                    raw_chunks.append(current_chunk.strip())
                    current_chunk = ""

                if len(exchange) > max_chars:
                    parts = exchange.split("\n")
                    for part in parts:
                        if (
                            len(current_chunk) + (1 if current_chunk else 0) + len(part)
                            <= max_chars
                        ):
                            current_chunk += ("\n" if current_chunk else "") + part
                        else:
                            if current_chunk:
                                raw_chunks.append(current_chunk.strip())
                            current_chunk = part
                else:
                    current_chunk = exchange

        if current_chunk:
            raw_chunks.append(current_chunk.strip())

        final_chunks = []
        for i, chunk in enumerate(raw_chunks):
            if i == 0:
                final_chunks.append(chunk)
            else:
                overlap_text = raw_chunks[i - 1][-overlap_size:]
                final_chunks.append(overlap_text + chunk)

        return final_chunks

    def _chunk_code(
        self, content: str, max_tokens: int, overlap_size: int
    ) -> List[str]:
        """
        Chunk code keeping block objects intact without breaking indentation.
        """
        max_chars = max_tokens * 4

        lines = content.split("\n")
        imports: List[str] = []
        blocks: List[str] = []
        current_block_lines: List[str] = []

        for line in lines:
            if line.startswith("import ") or line.startswith("from "):
                imports.append(line)
            elif line.startswith("def ") or line.startswith("class "):
                if current_block_lines:
                    blocks.append("\n".join(current_block_lines))
                current_block_lines = [line]
            else:
                if current_block_lines or line.strip():
                    current_block_lines.append(line)

        if current_block_lines:
            blocks.append("\n".join(current_block_lines))

        import_header = "\n".join(imports) + "\n\n" if imports else ""
        header_len = len(import_header)

        raw_chunks: List[str] = []
        current_chunk_blocks: List[str] = []
        current_chunk_len = header_len

        for block in blocks:
            block_len = len(block)

            if (
                current_chunk_len + (2 if current_chunk_blocks else 0) + block_len
                <= max_chars
            ):
                current_chunk_blocks.append(block)
                current_chunk_len += (
                    2 if len(current_chunk_blocks) > 1 else 0
                ) + block_len
            else:
                if current_chunk_blocks:
                    raw_chunks.append(import_header + "\n\n".join(current_chunk_blocks))
                    current_chunk_blocks = []
                    current_chunk_len = header_len

                if header_len + block_len > max_chars:
                    block_lines = block.split("\n")
                    for line in block_lines:
                        line_len = len(line) + 1
                        if current_chunk_len + line_len <= max_chars:
                            current_chunk_blocks.append(line)
                            current_chunk_len += line_len
                        else:
                            if current_chunk_blocks:
                                raw_chunks.append(
                                    import_header + "\n".join(current_chunk_blocks)
                                )
                            current_chunk_blocks = [line]
                            current_chunk_len = header_len + len(line)

                    if current_chunk_blocks:
                        remaining_block = "\n".join(current_chunk_blocks)
                        current_chunk_blocks = [remaining_block]
                        current_chunk_len = header_len + len(remaining_block)
                else:
                    current_chunk_blocks = [block]
                    current_chunk_len = header_len + block_len

        if current_chunk_blocks:
            raw_chunks.append(import_header + "\n\n".join(current_chunk_blocks))

        final_chunks: List[str] = []
        for i, chunk in enumerate(raw_chunks):
            if i == 0:
                final_chunks.append(chunk)
            else:
                prev_raw_content = raw_chunks[i - 1][header_len:]
                split_char = "\n\n" if "\n\n" in prev_raw_content else "\n"
                prev_parts = prev_raw_content.split(split_char)

                overlap_block = prev_parts[-1] if prev_parts else ""

                if overlap_block.strip():
                    final_chunks.append(
                        import_header + overlap_block + "\n\n" + chunk[header_len:]
                    )
                else:
                    final_chunks.append(chunk)

        return final_chunks

    def _chunk_markdown(
        self, content: str, max_tokens: int, overlap_size: int
    ) -> List[str]:
        max_chars = max_tokens * 4

        sections = []
        current_section_lines = []

        for line in content.split("\n"):
            stripped = line.lstrip()
            is_heading = stripped.startswith("#") and (
                len(stripped) == 1 or stripped[1] in (" ", "#")
            )

            if is_heading:
                if current_section_lines:
                    sections.append("\n".join(current_section_lines))
                current_section_lines = [line]
            else:
                current_section_lines.append(line)

        if current_section_lines:
            sections.append("\n".join(current_section_lines))

        raw_chunks = []
        current_chunk = ""

        for section_text in sections:
            if len(section_text) > max_chars:
                paragraphs = section_text.split("\n\n")
                for paragraph in paragraphs:
                    if (
                        len(current_chunk)
                        + (2 if current_chunk else 0)
                        + len(paragraph)
                        <= max_chars
                    ):
                        current_chunk += ("\n\n" if current_chunk else "") + paragraph
                    else:
                        if current_chunk:
                            raw_chunks.append(current_chunk.strip())
                        current_chunk = paragraph
            else:
                raw_chunks.append(section_text.strip())

        if current_chunk:
            raw_chunks.append(current_chunk.strip())

        final_chunks = []
        for i, chunk in enumerate(raw_chunks):
            if i == 0:
                final_chunks.append(chunk)
            else:
                overlap_text = raw_chunks[i - 1][-overlap_size:]
                final_chunks.append(overlap_text + chunk)

        return final_chunks

cc = ContentChunker()

content = """# Main Title

This is the introduction to our documentation.

## Getting Started

To get started, follow these steps.

### Installation

Run the following command to install:

```bash
pip install our-package
```

### Configuration

Create a configuration file with your settings.

## Usage

Here's how to use the main features.

### Basic Example

```python
from package import main
result = main()
```

## Advanced Topics

For advanced users, we provide additional features.

### Performance Tuning

Optimize your application for better performance.

## Conclusion

Thank you for using our package!
"""

chunks = cc.chunk_by_content_type(
    content,
    "markdown",
    max_tokens=200
)

chunks

['# Main Title\n\nThis is the introduction to our documentation.',
 'oduction to our documentation.## Getting Started\n\nTo get started, follow these steps.',
 't started, follow these steps.### Installation\n\nRun the following command to install:\n\n```bash\npip install our-package\n```',
 'sh\npip install our-package\n```### Configuration\n\nCreate a configuration file with your settings.',
 "ation file with your settings.## Usage\n\nHere's how to use the main features.",
 ' how to use the main features.### Basic Example\n\n```python\nfrom package import main\nresult = main()\n```',
 'mport main\nresult = main()\n```## Advanced Topics\n\nFor advanced users, we provide additional features.',
 'e provide additional features.### Performance Tuning\n\nOptimize your application for better performance.',
 'cation for better performance.## Conclusion\n\nThank you for using our package!']

In [ ]:
"""
Prompt Chain Module

This module implements sequential and branching prompt chains for complex
multi-step analysis workflows.

COMPLETE THE TODO SECTIONS IN THIS FILE.
"""

from typing import List, Dict
# from api_client import call_llm_safe


class PromptChain:
    """
    Executes complex multi-step prompt chains for sophisticated analysis.

    Supports two chain types:
    - Sequential: Each step uses output from the previous step
    - Branching: Parallel analysis paths that synthesize into final output
    """

    def __init__(self):
        """Initialize the PromptChain."""
        self.results = {}

    async def execute_sequential_chain(
        self,
        steps: List[Dict[str, str]],
        initial_input: str
    ) -> Dict[str, str]:
        """
        Execute a sequential chain where each step builds on the previous result.

        Sequential chains are ideal for linear workflows like:
        - Extract data → Analyze data → Summarize findings
        - Read document → Identify issues → Propose solutions
        - Parse code → Find bugs → Suggest fixes

        Each step's output becomes the input for the next step.

        Args:
            steps: List of step dictionaries with "name" and "prompt_template" keys
            initial_input: The starting input text

        Returns:
            Dictionary mapping step names to their outputs

        Example steps format:
            [
                {
                    "name": "extract",
                    "prompt_template": "Extract key points from: {previous_output}"
                },
                {
                    "name": "analyze",
                    "prompt_template": "Analyze these points: {previous_output}"
                }
            ]

        Hints:
        1. Initialize results dictionary
        2. Set previous_output = initial_input for first step
        3. For each step:
           a. Get the prompt template
           b. Format it with {previous_output} → use .format(previous_output=...)
           c. Call await call_llm_safe() with the formatted prompt
           d. Store result in results dictionary with step name as key
           e. Update previous_output for next step
        4. Return results dictionary

        Example:
            chain = PromptChain()
            steps = [
                {"name": "extract", "prompt_template": "Extract data from: {previous_output}"},
                {"name": "summarize", "prompt_template": "Summarize: {previous_output}"}
            ]
            results = chain.execute_sequential_chain(steps, "Document text...")
            # Returns: {"extract": "...", "summarize": "..."}
        """
        results = {}

        previous_output = initial_input
        for step in steps:
            prompt = step["prompt_template"].format(previous_output=previous_output)
            result = await call_llm_safe(prompt)
            results[step["name"]] = result
            previous_output = result

        return results

    async def execute_branching_chain(
        self,
        initial_prompt: str,
        branch_prompts: Dict[str, str],
        synthesis_prompt: str,
        input_text: str
    ) -> Dict[str, str]:
        """
        Execute a branching chain with parallel analysis paths and final synthesis.

        Branching chains are ideal for multi-perspective analysis:
        - Financial + Market + Operational analysis → Synthesis
        - Technical + Business + UX review → Final recommendation
        - Security + Performance + Maintainability audit → Summary

        Flow:
        1. Initial analysis extracts/structures data
        2. Multiple branches analyze different aspects in parallel
        3. Synthesis combines all branch findings

        Args:
            initial_prompt: Template for initial analysis (uses {input})
            branch_prompts: Dict mapping branch names to their prompt templates
            synthesis_prompt: Template for final synthesis (uses {branch_name_output})
            input_text: The starting input text

        Returns:
            Dictionary with "initial", branch names, and "synthesis" keys

        Example:
            chain = PromptChain()
            results = chain.execute_branching_chain(
                initial_prompt="Extract data from: {input}",
                branch_prompts={
                    "financial": "Analyze finances: {initial_output}",
                    "market": "Analyze market: {initial_output}"
                },
                synthesis_prompt="Synthesize: {financial_output} and {market_output}",
                input_text="Company data..."
            )
            # Returns: {
            #     "initial": "...",
            #     "financial": "...",
            #     "market": "...",
            #     "synthesis": "..."
            # }

        Hints:
        1. Initialize results dictionary
        2. Execute initial step:
           a. Format initial_prompt with {input} → initial_prompt.format(input=input_text)
           b. Call await call_llm_safe()
           c. Store in results["initial"]
        3. Execute each branch:
           a. For each branch_name, prompt_template in branch_prompts.items()
           b. Format template with {initial_output} → prompt.format(initial_output=results["initial"])
           c. Call await call_llm_safe()
           d. Store in results[branch_name]
        4. Execute synthesis:
           a. Build kwargs dict: {f"{branch}_output": results[branch] for branch in branches}
           b. Format synthesis_prompt with all branch outputs
           c. Call await call_llm_safe()
           d. Store in results["synthesis"]
        5. Return results

        Note: synthesis_prompt uses {branch_name_output} placeholders
        For branch "financial", use {financial_output}
        """
        results = {}

        initial_prompt.format(input=input_text)
        results["initial"] = await call_llm_safe(initial_prompt)

        for branch_name, prompt_template in branch_prompts.items():
            prompt = prompt_template.format(initial_output=results["initial"])
            results[branch_name] = await call_llm_safe(prompt)

        synthesis_kwargs = {f"{branch}_output": results[branch] for branch in branch_prompts}
        synthesis_prompt = synthesis_prompt.format(**synthesis_kwargs)
        results["synthesis"] = await call_llm_safe(synthesis_prompt)

        return results


ModuleNotFoundError: No module named 'api_client'